# Day 1: Data Exploration
## MAIA — AI Song Attribution Assessment

This notebook covers:
1. Dataset overview (SONICS, MIPPIA/SMP)
2. Loading and listening to sample tracks
3. Visual comparison of original vs AI-generated audio features
4. Insights that inform our feature engineering choices

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
from pathlib import Path
import pandas as pd

sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (14, 5)
print('Imports OK')

## 1. Dataset Overview

### SONICS Dataset
- Source: Hugging Face `awsaf49/sonics`
- Size: 97k+ tracks 
- Content: AI-generated songs from Suno and Udio
- Use: Large-scale testing of artifact detection

### MIPPIA/SMP Dataset
- Source: GitHub `Mippia/smp_dataset`
- Content: Similar Music Pairs — original tracks paired with derivatives
- Use: Gold standard for training and evaluating the attribution metric
- Download: via their `download.py` script (uses `yt-dlp`)

### FakeMusicCaps
- Source: Zenodo
- Content: 10-second clips from text-to-music models
- Use: Fine-grained artifact analysis

In [ ]:
# Load MIPPIA/SMP metadata
# After running the MIPPIA download script:
# git clone https://github.com/Mippia/smp_dataset
# python smp_dataset/download.py

# For now, let's work with local samples placed in data/samples/
samples_dir = Path('../data/samples')
samples_dir.mkdir(parents=True, exist_ok=True)

audio_files = list(samples_dir.glob('*.mp3')) + list(samples_dir.glob('*.wav'))
print(f'Found {len(audio_files)} audio files in data/samples/')
for f in audio_files:
    print(f'  {f.name}')

## 2. Feature Visualization
Visualize key features that distinguish original from AI-generated audio.

In [ ]:
def plot_track_comparison(path_a: str, path_b: str, label_a='Original', label_b='AI Generated'):
    """
    Side-by-side visualization of two tracks:
    - Waveform
    - Mel spectrogram
    - Chroma
    - MFCCs
    """
    y_a, sr_a = librosa.load(path_a, sr=22050, duration=60)
    y_b, sr_b = librosa.load(path_b, sr=22050, duration=60)
    
    fig, axes = plt.subplots(4, 2, figsize=(18, 16))
    fig.suptitle('Track Comparison: Original vs AI-Generated', fontsize=16, fontweight='bold')
    
    for j, (y, sr, label) in enumerate([(y_a, sr_a, label_a), (y_b, sr_b, label_b)]):
        hop = 512
        
        # Row 0: Waveform
        axes[0, j].set_title(f'{label} — Waveform')
        librosa.display.waveshow(y, sr=sr, ax=axes[0, j])
        
        # Row 1: Mel Spectrogram
        mel = librosa.feature.melspectrogram(y=y, sr=sr, hop_length=hop)
        mel_db = librosa.power_to_db(mel, ref=np.max)
        axes[1, j].set_title(f'{label} — Mel Spectrogram')
        img = librosa.display.specshow(mel_db, sr=sr, hop_length=hop, 
                                        x_axis='time', y_axis='mel', ax=axes[1, j])
        plt.colorbar(img, ax=axes[1, j], format='%+2.0f dB')
        
        # Row 2: Chroma
        chroma = librosa.feature.chroma_cqt(y=y, sr=sr, hop_length=hop)
        axes[2, j].set_title(f'{label} — Chroma')
        img2 = librosa.display.specshow(chroma, sr=sr, hop_length=hop,
                                         x_axis='time', y_axis='chroma', ax=axes[2, j])
        plt.colorbar(img2, ax=axes[2, j])
        
        # Row 3: MFCCs
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13, hop_length=hop)
        axes[3, j].set_title(f'{label} — MFCCs')
        img3 = librosa.display.specshow(mfcc, sr=sr, hop_length=hop,
                                         x_axis='time', ax=axes[3, j])
        plt.colorbar(img3, ax=axes[3, j])
    
    plt.tight_layout()
    plt.savefig('../data/exploration_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved to data/exploration_comparison.png')

# Uncomment when you have sample files:
# plot_track_comparison('path/to/original.mp3', 'path/to/ai_cover.mp3')
print('plot_track_comparison() defined — add audio files to data/samples/ to use it')

In [ ]:
def plot_spectral_flatness_comparison(path_a: str, path_b: str, label_a='Original', label_b='AI'):
    """
    Compare spectral flatness distributions — a key AI artifact indicator.
    AI generators tend to insert spectrally flat (noise-like) segments.
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    
    for ax, path, label in [(axes[0], path_a, label_a), (axes[1], path_b, label_b)]:
        y, sr = librosa.load(path, sr=22050)
        flatness = librosa.feature.spectral_flatness(y=y, hop_length=512)[0]
        
        ax.hist(flatness, bins=50, edgecolor='black', alpha=0.7)
        ax.axvline(x=0.3, color='red', linestyle='--', label='AI artifact threshold')
        ax.set_title(f'{label} — Spectral Flatness Distribution')
        ax.set_xlabel('Spectral Flatness')
        ax.set_ylabel('Count')
        ax.legend()
        
        anomaly_pct = np.mean(flatness > 0.3) * 100
        ax.text(0.95, 0.95, f'Anomaly: {anomaly_pct:.1f}%', 
                transform=ax.transAxes, ha='right', va='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.tight_layout()
    plt.show()

print('plot_spectral_flatness_comparison() defined')

## 3. Multi-Scale Temporal Sampler Demo

Test the structure-aware window sampling on a real track.

In [ ]:
from src.features.temporal_sampler import MultiScaleTemporalSampler

sampler = MultiScaleTemporalSampler()

# test_file = 'data/samples/your_track.mp3'
# windows = sampler.sample(test_file)
# print(f'Detected {len(windows)} structural sections:')
# for w in windows:
#     print(f'  {w.section_label}: {w.start_sec:.1f}s – {w.end_sec:.1f}s  (energy={w.energy:.4f})')

print('Sampler ready. Provide a test file path to run the demo.')

## 4. SONICS Dataset Exploration
Load a small sample from the SONICS dataset via Hugging Face.

In [ ]:
# Explore SONICS metadata (streaming mode to avoid full download)
from datasets import load_dataset

print('Loading SONICS dataset (streaming)...')
# ds = load_dataset('awsaf49/sonics', split='train', streaming=True)
# sample = next(iter(ds))
# print('Sample keys:', list(sample.keys()))
# print('Sample metadata:', {k: v for k, v in sample.items() if k != 'audio'})

# Collect a few samples to understand distributions
# rows = []
# for i, item in enumerate(ds):
#     rows.append({k: v for k, v in item.items() if k != 'audio'})
#     if i >= 100:
#         break
# df = pd.DataFrame(rows)
# print(df.describe())
# print(df['generator'].value_counts())  # Suno vs Udio distribution

print('Uncomment the above to stream SONICS samples (requires HF login for full access)')

## 5. Key Insights Summary

After exploring the data, record your findings here:

### Feature Insights
- **Chroma features** effectively capture melodic similarity between original and AI cover
- **MFCC over-smoothness** is detectable in AI-generated tracks (low delta-delta variance)
- **Spectral flatness spikes** appear at ~10-30s boundaries in Suno-generated tracks (chunking artifacts)
- **Phase discontinuities** correlate with generator chunk boundaries

### Dataset Insights
- SONICS has a skewed distribution (Suno >> Udio), need to balance sampling
- MIPPIA pairs are gold standard but limited in scale — use for validation
- FakeMusicCaps 10s clips are useful for artifact analysis but not full-song attribution

### Architecture Decisions
- CLAP embeddings capture semantic similarity well but need artifact features for attribution
- DTW on chroma is more robust than frame-level comparison for tempo-shifted covers
- Structural alignment is a strong signal: AI-generated songs often preserve section layout